# Notebook 03: Protein Folding & Peptide Design Demo

This tutorial demonstrates:
1. Predicting 3D target structure and pLDDT confidence using the **AlphaFoldRunner**.
2. Generating per-residue confidence profiles.
3. Running **BindCraft** to generate targeted de novo mini-protein binders.
4. Designing macrocyclic peptides (**RFpeptides**) and antimicrobial peptides (**AMPGen**).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath("../src"))

import matplotlib.pyplot as plt
from protein_design.alphafold_runner import AlphaFoldRunner
from protein_design.bindcraft_wrapper import BindCraftWrapper
from protein_design.peptide_generator import PeptideGenerator
from utils.viz_utils import plot_plddt_profile

## 1. Fold Target Protein / Peptide with AlphaFold

In [ ]:
af = AlphaFoldRunner(model_version="AlphaFold3", num_recycles=3)
target_seq = "MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEEHFKGLVLIAFSQYLQQCPFDEHVKLVNELTEFAKTCVADESHAGCEKSLHTLF"
fold_result = af.predict_structure(target_seq, target_name="NTRK1_Kinase_Domain")

print(f"Target: {fold_result['target_name']}")
print(f"Sequence Length: {fold_result['sequence_length']} aa")
print(f"Mean pLDDT: {fold_result['mean_plddt']}")
print(f"High-Confidence Residues: {fold_result['high_confidence_percent']}%")
print(f"Is Druggable Fold: {fold_result['is_druggable_fold']}")

## 2. Visualize Structural Confidence Profile

In [ ]:
fig = plot_plddt_profile(fold_result["plddt_scores"], target_name="NTRK1 Kinase Domain")
plt.show()

## 3. De Novo Binder Design with BindCraft

In [ ]:
bindcraft = BindCraftWrapper(target_hotspots=["ARG120", "GLU154", "TRP210"], binder_length=70, num_iterations=10)
binders = bindcraft.design_binders()

print(f"Generated {len(binders)} candidate mini-protein binders.")
top_binder = binders[0]
print("Top Scoring Binder:")
for k, v in top_binder.items():
    print(f"  {k}: {v}")

## 4. Macrocycles (RFpeptides) and AMPGen Peptides

In [ ]:
pep_gen = PeptideGenerator(random_seed=42)
cyclic_pep = pep_gen.generate_cyclic_peptide(length=12, cyclization_type="head_to_tail")
print("Designed Cyclic Macrocycle:")
print(cyclic_pep)

amp = pep_gen.generate_antimicrobial_peptide(length=18)
print("\nDesigned Antimicrobial Peptide (AMPGen):")
print(amp)